**Migration status (2026-11-18):** this is a path-adapted working copy of Timothy Tran's inherited notebook. Saved outputs/execution counters were cleared to avoid implying a new run. The original notebook and inherited outputs remain byte-identical in `archive/original_pilot`. Historical narrative below describes the handoff, not a verified migration run. Raw input is now `data/raw/freshretail.csv`; all generated outputs go to separate `reproduction` directories. This notebook was syntax/JSON checked, not executed. It is not the planned matched-feature experiment.

# DATA 245 – Stockout-Aware Forecasting for Fresh Retail
## Dataset Preparation and Baseline Forecasting

This project asks whether stockouts can make strong fresh-retail products appear unpopular. My responsibility is data validation and preparation, a reproducible pilot, chronological splits, simple forecasting baselines, and evaluation. Using FreshRetailNet-50K, this notebook prepares the pilot and benchmarks persistence and a seven-day recent average before the group runs its regression and tree models.

## 1. Imports and Configuration

Paths are rooted at the repository, and the pilot design is centralized here for reproducibility.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / 'AGENTS.md').is_file() and (path / 'pyproject.toml').is_file())
DATA_PATH = ROOT / 'data' / 'raw' / 'freshretail.csv'
PROCESSED_DIR = ROOT / 'data' / 'processed' / 'reproduction'
MANIFESTS_DIR = ROOT / 'data' / 'manifests' / 'reproduction'
RESULTS_DIR = ROOT / 'results' / 'baselines_DELTA' / 'reproduction'
RANDOM_SEED = 245
N_PILOT_SERIES = 200
EXPECTED_DAYS = 90
SERIES_KEYS = ['store_id', 'product_id']
RECORD_KEYS = [*SERIES_KEYS, 'dt']
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load the Dataset

The source CSV is parsed with `dt` as a date. The preview and schema summary keep the large dataset readable.

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['dt'])
required = {'store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt'}
missing = required.difference(df.columns)
assert not missing, f'Missing required columns: {sorted(missing)}'
assert df['dt'].notna().all()
display(df.head())
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df.dt.min().date()} through {df.dt.max().date()}')
print('Columns:', ', '.join(df.columns))

## 3. Data Validation and Audit

Audit counts and series coverage are computed from the dataset. Completeness is based on the actual expected date set, not row count alone.

In [ ]:
expected_dates = pd.date_range(df.dt.min(), df.dt.max(), freq='D')
assert len(expected_dates) == EXPECTED_DAYS and df.dt.nunique() == EXPECTED_DAYS
null_counts = df.isna().sum()
duplicate_count = int(df.duplicated(RECORD_KEYS).sum())
series_lengths = df.groupby(SERIES_KEYS, observed=True).size().rename('row_count')
series_dates = df.groupby(SERIES_KEYS, observed=True).dt.agg(lambda s: frozenset(s))
expected_date_set = frozenset(expected_dates)
complete_mask = series_dates.eq(expected_date_set)
complete_index = series_dates.index[complete_mask]
incomplete_index = series_dates.index[~complete_mask]
length_distribution = series_lengths.value_counts().sort_index()
audit = pd.DataFrame({'measure':['Rows','Columns','Date range','Distinct dates','Store-product series','Duplicate series-date rows',f'Complete {EXPECTED_DAYS}-date series','Incomplete series'], 'value':[len(df),len(df.columns),f'{df.dt.min().date()} to {df.dt.max().date()}',df.dt.nunique(),len(series_lengths),duplicate_count,int(complete_mask.sum()),int((~complete_mask).sum())]})
display(audit)
print('Series length distribution:'); display(length_distribution.rename_axis('rows_per_series').rename('series_count').to_frame())
print('Null counts by column:'); display(null_counts.rename('null_count').to_frame())
assert null_counts.sum() == 0 and duplicate_count == 0

The current dataset is very clean, with no null values or duplicate store-product-date records. One series has only 28 days, so it should be excluded before lagged features and chronological modeling.

## 4. Chronological Ordering

Sorting within each store-product series ensures that lags and forecasts refer only to earlier observations.

In [ ]:
df = df.sort_values(RECORD_KEYS, kind='mergesort').reset_index(drop=True)
chronological = df.groupby(SERIES_KEYS, sort=False, observed=True).dt.apply(lambda s: s.is_monotonic_increasing)
assert chronological.all()
print(f'Chronological order verified for {len(chronological):,} series.')

## 5. Complete-Series Filtering

Each retained series must contain all 90 expected calendar dates. Incomplete series are saved with their missing-date details.

In [ ]:
excluded_rows = []
for (store_id, product_id), dates in series_dates.loc[~complete_mask].items():
    excluded_rows.append({'store_id':store_id, 'product_id':product_id, 'row_count':int(series_lengths.loc[(store_id,product_id)]), 'distinct_dates':len(dates), 'missing_dates':';'.join(d.strftime('%Y-%m-%d') for d in expected_dates if d not in dates), 'unexpected_dates':';'.join(sorted(d.strftime('%Y-%m-%d') for d in dates - expected_date_set))})
excluded = pd.DataFrame(excluded_rows, columns=['store_id','product_id','row_count','distinct_dates','missing_dates','unexpected_dates'])
excluded.to_csv(MANIFESTS_DIR / 'excluded_series.csv', index=False)
complete_series = pd.DataFrame(complete_index.tolist(), columns=SERIES_KEYS)
print(f'Complete series: {len(complete_series):,}; incomplete series: {len(excluded):,}')
display(excluded)

The incomplete series is excluded from pilot sampling. The separate CSV preserves the exact exclusion details for review.

## 6. Reproducible 200-Series Pilot

Sampling occurs at the store-product level with a fixed seed, so reruns select the same pairs.

In [ ]:
assert len(complete_series) >= N_PILOT_SERIES
manifest = complete_series.sample(n=N_PILOT_SERIES, random_state=RANDOM_SEED).sort_values(SERIES_KEYS).reset_index(drop=True)
manifest.to_csv(MANIFESTS_DIR / 'pilot_series_manifest.csv', index=False)
pilot = df.merge(manifest, on=SERIES_KEYS, how='inner', validate='many_to_one').sort_values(RECORD_KEYS, kind='mergesort').reset_index(drop=True)
expected_pilot_rows = N_PILOT_SERIES * EXPECTED_DAYS
assert len(pilot) == expected_pilot_rows
print(f'Sampled series: {len(manifest):,}; pilot rows: {len(pilot):,} (expected {expected_pilot_rows:,})')

## 7. Day Number and Chronological Splits

Shared calendar dates map to the same day number for every series. Chronological splits preserve temporal order and avoid random train/test shuffling.

In [ ]:
date_to_day = {date: day for day, date in enumerate(expected_dates, start=1)}
pilot['day_number'] = pilot.dt.map(date_to_day).astype('int8')
assert pilot.groupby('dt').day_number.nunique().eq(1).all()
pilot['split'] = pd.cut(pilot.day_number, bins=[0,60,75,90], labels=['train','validation','test']).astype('string')
split_summary = pilot.groupby('split', observed=False).agg(start_date=('dt','min'), end_date=('dt','max'), rows=('dt','size')).reset_index()
split_summary['start_date'] = split_summary.start_date.dt.date
split_summary['end_date'] = split_summary.end_date.dt.date
display(split_summary)

## 8. Persistence Baseline

For each store-product series, the persistence forecast for day *t* is the sale amount from day *t ? 1*. The first observation in each series has no valid prediction and is excluded from evaluation.

In [ ]:
sales = pilot.groupby(SERIES_KEYS, sort=False, observed=True)['sale_amount']
pilot['pred_persistence'] = sales.shift(1)


## 9. Seven-Day Recent-Average Baseline

For each series, the forecast for day *t* is the mean sale amount from days *t ? 7* through *t ? 1*. Shifting the series before applying the rolling mean ensures the target day is excluded.

In [ ]:
pilot['pred_recent_avg_7'] = sales.transform(
    lambda s: s.shift(1).rolling(window=7, min_periods=7).mean()
)
pilot.to_csv(PROCESSED_DIR / 'pilot_200_series.csv', index=False)
display(pilot[[*SERIES_KEYS, 'dt', 'day_number', 'sale_amount', 'pred_persistence', 'pred_recent_avg_7']].head(10))


## 10. Evaluation Metrics

Reusable functions calculate MAE, RMSE, and signed bias, defined as `mean(predicted ? actual)`. Positive bias indicates overprediction; negative bias indicates underprediction. Evaluation is limited to validation and test, with unavailable predictions excluded.

In [ ]:
def calculate_metrics(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    error = predicted - actual
    return {'MAE':error.abs().mean(), 'RMSE':np.sqrt(error.pow(2).mean()), 'bias':error.mean()}



## 11. Baseline Results

The following table compares both baselines separately on validation and test. Training metrics are not presented as primary results.

In [ ]:
metric_rows, prediction_frames = [], []

for model, pred_col in [('persistence','pred_persistence'), ('recent_avg_7','pred_recent_avg_7')]:
    for split in ['validation','test']:
        scored = pilot.loc[pilot.split.eq(split) & pilot[pred_col].notna(), [*SERIES_KEYS,'dt','day_number','split','sale_amount',pred_col]].copy()
        metric_rows.append({'model':model, 'split':split, 'n_observations':len(scored), **calculate_metrics(scored.sale_amount, scored[pred_col])})
        scored = scored.rename(columns={pred_col:'prediction'}); scored.insert(0,'model',model); prediction_frames.append(scored)
baseline_metrics = pd.DataFrame(metric_rows)
baseline_predictions = pd.concat(prediction_frames, ignore_index=True)
baseline_predictions.to_csv(RESULTS_DIR / 'baseline_predictions.csv', index=False)
baseline_metrics.to_csv(RESULTS_DIR / 'baseline_metrics.csv', index=False, float_format='%.6f')
display(baseline_metrics.round(6))

## 12. Preliminary Interpretation

The seven-day recent-average baseline has lower MAE and RMSE than persistence on both validation and test. Persistence has near-zero test bias; the seven-day average has more negative test bias, indicating some underprediction. These are baseline results only and do not yet answer whether stockout-aware features improve forecasting. The group will compare stockout-aware with sales-only inputs in its later regression and tree models.

## 13. Leakage Prevention and Reproducibility Checks

These implementation checks are not a formal leakage audit. They verify chronological order, prior-only forecasts, the fixed seed, common date splits, and the absence of random row shuffling.

In [ ]:
assert chronological.all() and RANDOM_SEED == 245 and len(manifest) == N_PILOT_SERIES
assert pilot.groupby('dt').day_number.nunique().eq(1).all()
assert pilot.groupby(SERIES_KEYS, observed=True).split.nunique().eq(3).all()
assert pilot.pred_persistence.isna().equals(pilot.groupby(SERIES_KEYS, observed=True).cumcount().eq(0))
assert pilot.groupby(SERIES_KEYS, observed=True).pred_recent_avg_7.apply(lambda s: s.iloc[:7].isna().all() and s.iloc[7:].notna().all()).all()
# Recompute an example directly to verify the target day is excluded.
example = pilot.groupby(SERIES_KEYS, sort=False, observed=True).head(10).groupby(SERIES_KEYS, sort=False, observed=True).head(10).iloc[:10]
sales_example = example.sale_amount.to_numpy()
np.testing.assert_allclose(example.pred_persistence.iloc[1:].to_numpy(), sales_example[:-1])
np.testing.assert_allclose(example.pred_recent_avg_7.iloc[7:].to_numpy(), [sales_example[i-7:i].mean() for i in range(7, len(sales_example))])
print('Chronology, prior-only forecasts, fixed-seed sampling, and common splits verified.')

## 14. Generated Output Files

The pipeline writes the following reproducible artifacts:

In [ ]:
# Keep the audit report in sync with the data computed above.
audit_lines = ['FreshRetail data audit', f'Rows: {len(df):,}', f'Columns: {len(df.columns)}', f'Date range: {df.dt.min().date()} through {df.dt.max().date()}', f'Distinct dates: {df.dt.nunique()}', f'Unique store-product series: {len(series_lengths):,}', f'Duplicate (store_id, product_id, dt) rows: {duplicate_count}', f'Complete {EXPECTED_DAYS}-date series: {int(complete_mask.sum()):,}', f'Incomplete series: {int((~complete_mask).sum()):,}', 'Series-length distribution (rows per series: series count):']
audit_lines += [f'  {int(k)}: {int(v):,}' for k,v in length_distribution.items()]
audit_lines += ['Null count by column:'] + [f'  {col}: {int(value):,}' for col,value in null_counts.items()]
audit_lines += ['Incomplete series details (also saved to data/manifests/reproduction/excluded_series.csv):']
audit_lines += [f'  store={r.store_id}, product={r.product_id}, rows={r.row_count}, distinct_dates={r.distinct_dates}, missing={r.missing_dates or "none"}' for r in excluded.itertuples(index=False)] or ['  None']
(RESULTS_DIR / 'data_audit.txt').write_text('\n'.join(audit_lines) + '\n', encoding='utf-8')
outputs = ['data/manifests/reproduction/excluded_series.csv','data/manifests/reproduction/pilot_series_manifest.csv','data/processed/reproduction/pilot_200_series.csv','results/baselines_DELTA/reproduction/data_audit.txt','results/baselines_DELTA/reproduction/baseline_predictions.csv','results/baselines_DELTA/reproduction/baseline_metrics.csv']
print('\n'.join(outputs))

## 15. Conclusion

Dataset validation is complete, the reproducible 200-series pilot is prepared, and chronological train/validation/test splits are established. Baseline performance has been measured. These outputs are ready for the group?s later sales-only versus stockout-aware model comparison.